# Custom PrimitiveSet + Symbolic Export — Schwefel 2D
**Surrogate-assisted optimization** with a domain-specific custom primitive set.

New library features showcased (not used in other notebooks):
- `ARITHMETIC` — minimal preset with only add, sub, mul, div
- `add_custom(..., sympy_fn=...)` — register custom primitives with symbolic export
- `to_sympy()` — tree-traversal conversion to sympy (works with custom functions)
- `to_latex()` — LaTeX rendering of the symbolic expression
- `to_callable()` — standalone Python callable for deployment

**Benchmark — Schwefel 2D:**
$$f(x_1, x_2) = 418.9829 \cdot 2 - x_1 \sin\!\left(\sqrt{|x_1|}\right) - x_2 \sin\!\left(\sqrt{|x_2|}\right)$$
Domain: $[-500, 500]^2$, global minimum at $(420.97, 420.97)$, $f=0$.
Highly multimodal — many deceptive local minima far from the global optimum.

Custom pset primitives:

| Name | Formula | Role |
|---|---|---|
| `sqrtabs` | $\sqrt{|x|}$ | Intermediate Schwefel component |
| `xsinqrt` | $x\sin(\sqrt{|x|})$ | Core Schwefel building block |

Figures produced:
- `fig05_schwefel_obs_pred.png` — Surrogate quality: Observed × Predicted
- `fig05_schwefel_surface.png` — 3D: true Schwefel vs MGGP surrogate
- `fig05_schwefel_contour.png` — PSO spatial distribution: true vs surrogate
- `fig05_schwefel_pso_convergence.png` — PSO convergence on the surrogate

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from pathlib import Path
from sklearn.metrics import r2_score
import sympy as sp
import os, math

FIG_DIR = Path("figures")
os.makedirs(FIG_DIR, exist_ok=True)

TRAIN_COLOR = '#1f77b4'
VAL_COLOR   = '#ff7f0e'
TEST_COLOR  = '#2ca02c'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.linestyle': '-', 'grid.alpha': 0.4,
    'grid.color': '#cccccc', 'font.size': 11, 'axes.labelsize': 12,
    'axes.titlesize': 13, 'legend.fontsize': 10,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

def obs_pred_ax(ax, y_tr, yp_tr, y_v, yp_v, y_te, yp_te, title,
                xlabel='Observed', ylabel='Predicted'):
    h1 = ax.scatter(y_tr, yp_tr, c=TRAIN_COLOR, marker='o', s=40, alpha=0.75,
                    label=f'Train  (R²={r2_score(y_tr, yp_tr):.6f})')
    h2 = ax.scatter(y_v,  yp_v,  c=VAL_COLOR,   marker='s', s=40, alpha=0.75,
                    label=f'Val    (R²={r2_score(y_v,  yp_v):.6f})')
    h3 = ax.scatter(y_te, yp_te, c=TEST_COLOR,  marker='^', s=40, alpha=0.75,
                    label=f'Test   (R²={r2_score(y_te, yp_te):.6f})')
    all_y = np.concatenate([y_tr, y_v, y_te])
    lo, hi = all_y.min(), all_y.max()
    ax.plot([lo, hi], [lo, hi], color='black', lw=1.5)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    return h1, h2, h3

print('Style loaded.')

def _snap(ax, log_y=False):
    ax.figure.canvas.draw()
    ax.tick_params(top=True, right=True, which='both', direction='in')
    lo, hi = ax.get_xlim()
    xt = sorted(t for t in ax.get_xticks() if lo - 1e-9 <= t <= hi + 1e-9)
    if len(xt) >= 2:
        ax.set_xlim(xt[0], xt[-1])
    if log_y:
        lo, hi = ax.get_ylim()
        if lo > 0 and hi > 0:
            lo_dec = 10 ** math.floor(math.log10(lo))
            hi_log = math.log10(hi)
            frac   = hi_log - math.floor(hi_log)
            hi_dec = 10 ** (math.floor(hi_log) if frac < 0.02 else math.ceil(hi_log))
            if lo_dec < hi_dec:
                ax.set_ylim(lo_dec, hi_dec)
    else:
        lo, hi = ax.get_ylim()
        yt = sorted(t for t in ax.get_yticks() if lo - 1e-9 <= t <= hi + 1e-9)
        if len(yt) >= 2:
            ax.set_ylim(yt[0], yt[-1])

def _fix_cbar(cb, cp):
    _lo, _hi = cp.get_clim()
    _t = [x for x in cb.get_ticks() if _lo < x < _hi]
    cb.set_ticks([_lo] + _t + [_hi])


## Data Generation (70/15/15 split from 400 LHS samples)

In [3]:
from symgene.benchmarks import schwefel_2d

bench = schwefel_2d()
rng   = np.random.default_rng(0)

def lhs(rng, lo, hi, n):
    d = len(lo)
    X = np.empty((n, d))
    for j in range(d):
        perm = rng.permutation(n)
        X[:, j] = lo[j] + (perm + rng.uniform(0, 1, n)) / n * (hi[j] - lo[j])
    return X

lo, hi = np.array([-500., -500.]), np.array([500., 500.])
X_all  = lhs(rng, lo, hi, 400)
y_all  = np.array([bench.fn(X_all[i]) for i in range(400)])

n_train, n_val = 280, 60           # 70 / 15 / 15
X_train, y_train = X_all[:n_train], y_all[:n_train]
X_val,   y_val   = X_all[n_train:n_train + n_val], y_all[n_train:n_train + n_val]
X_test,  y_test  = X_all[n_train + n_val:], y_all[n_train + n_val:]

print(f'Train:{X_train.shape[0]}  Val:{X_val.shape[0]}  Test:{X_test.shape[0]}')
print(f'y range: [{y_all.min():.2f}, {y_all.max():.2f}]')

Train:280  Val:60  Test:60
y range: [6.02, 1671.72]


## Fit MGGP Surrogate — Custom PrimitiveSet

| Parameter | Value | Description |
|---|---|---|
| `primitives` | `ARITHMETIC + ["sin", "cos"]` | Minimal catalog selection |
| `add_custom("sqrtabs", sympy_fn=...)` | $\sqrt{|x|}$ | Intermediate Schwefel component |
| `add_custom("xsinqrt", sympy_fn=...)` | $x\sin(\sqrt{|x|})$ | Core Schwefel building block |
| `combiner` | `ridge` | Ridge regularisation |
| `regression_degree` | `1` | Linear combination of gene outputs |

In [4]:
from symgene import PrimitiveSet, Population, SymGeneEvolver
from symgene.primitives import ARITHMETIC
from symgene.fitness import FitnessEvaluator
from symgene.metrics import mae, r2, mape
from symgene.metrics.regression import mse
from symgene.selection import TournamentSelection

pset = PrimitiveSet(n_inputs=2, feature_names=['x1', 'x2'])
pset.set_squash(lim=800, alpha=0.01, scale=2.0)
pset.add_from_catalog(ARITHMETIC + ['sin', 'cos'])
pset.add_custom(
    fn=lambda x: float(np.sqrt(abs(x))),
    arity=1,
    name='sqrtabs',
    sympy_fn=lambda x: sp.sqrt(sp.Abs(x)),              # <<< sympy_fn
)
pset.add_custom(
    fn=lambda x: float(x * np.sin(np.sqrt(abs(x)))),
    arity=1,
    name='xsinqrt',
    sympy_fn=lambda x: x * sp.sin(sp.sqrt(sp.Abs(x))),  # <<< sympy_fn
)

pop = Population(
    name='schwefel',
    pset=pset,
    n_genes=4,
    pop_size=60,
    combiner='ridge',
    ridge_alphas=[0.01, 0.1, 1.0, 10.0, 100.0],
    regression_degree=1,
    fitness=FitnessEvaluator(metric=mse),
    selection=TournamentSelection(size=7),
    mutpb=0.25, mutpb_low=0.30,
    mutation_weights=[0.5, 1.5, 1.0],
)
evolver = SymGeneEvolver(populations=[pop], n_gen=80, seed=0, verbose=0)
results = evolver.fit(
    X_train, {'schwefel': y_train},
    X_val=X_val, y_val={'schwefel': y_val},
)
regressor = results['schwefel']
print('MGGP surrogate fitted.')
print(f'Genes     : {regressor.n_genes_}')
print(f'Expression: {regressor.best_expression_}')

MGGP surrogate fitted.
Genes     : 6
Expression: xsinqrt(x1) | sqrtabs(x2) | xsinqrt(x2) | sqrtabs(sub(x1, x1)) | sin(sqrtabs(div(cos(xsinqrt(sub(sin(cos(x1)), add(div(x1, x1), mul(x2, x2))))), add(sub(sqrtabs(add(sub(x1, x2), sqrtabs(x1))), div(mul(sqrtabs(x1), xsinqrt(x1)), add(sqrtabs(x1), div(x2, x2)))), sin(sin(sub(sub(x1, x2), add(x2, x2)))))))) | mul(sub(div(sin(cos(sqrtabs(div(mul(x2, x2), sqrtabs(x1))))), cos(sub(sub(div(div(x1, x1), sub(x1, x1)), sin(sin(x1))), sub(add(mul(x2, x2), sub(x2, x1)), add(xsinqrt(x1), sub(x1, x2)))))), mul(cos(cos(sqrtabs(mul(div(x2, x1), sub(x2, x1))))), mul(sin(sin(xsinqrt(mul(x2, x1)))), mul(sqrtabs(add(sub(x1, x2), xsinqrt(x2))), sqrtabs(sin(add(x1, x2))))))), mul(xsinqrt(sub(sin(mul(mul(add(x2, x1), xsinqrt(x1)), mul(sin(x1), mul(x2, x2)))), xsinqrt(sub(div(sqrtabs(x1), sub(x2, x2)), xsinqrt(mul(x1, x1)))))), div(cos(sqrtabs(cos(sub(sub(x1, x1), sub(x1, x1))))), sqrtabs(div(add(add(xsinqrt(x2), xsinqrt(x1)), div(x1, add(x1, x1))), sin(mul(sub(

## Symbolic Export — `to_sympy()` / `to_latex()` / `to_callable()`

Because both custom primitives declared `sympy_fn`, the full expression is renderable as LaTeX
and exportable as a standalone Python callable — without rebuilding any MGGP infrastructure.

In [5]:
from IPython.display import display, Math

latex_expr = regressor.to_latex()
print('LaTeX expression:')
print(latex_expr)
display(Math(latex_expr[:400]))

fn_callable = regressor.to_callable()
x_opt = np.array([[420.9687, 420.9687]])
print(f'\nto_callable() at optimum (420.97, 420.97):')
print(f'  Surrogate prediction : {fn_callable(x_opt)[0]:.4f}')
print(f'  True f at optimum    : {bench.fn(x_opt[0]):.6f}')

LaTeX expression:
ARG_{0} \sin{\left(\sqrt{\left|{ARG_{0}}\right|} \right)} + \sqrt{\left|{ARG_{1}}\right|} + ARG_{1} \sin{\left(\sqrt{\left|{ARG_{1}}\right|} \right)} + 0 + \sin{\left(\sqrt{\left|{\frac{\cos{\left(\left(ARG_{1}^{2} - \sin{\left(\cos{\left(ARG_{0} \right)} \right)} + 1\right) \sin{\left(\sqrt{\left|{ARG_{1}^{2} - \sin{\left(\cos{\left(ARG_{0} \right)} \right)} + 1}\right|} \right)} \right)}}{- \frac{ARG_{0} \sin{\left(\sqrt{\left|{ARG_{0}}\right|} \right)} \sqrt{\left|{ARG_{0}}\right|}}{\sqrt{\left|{ARG_{0}}\right|} + 1} + \sin{\left(\sin{\left(ARG_{0} - 3 ARG_{1} \right)} \right)} + \sqrt{\left|{ARG_{0} - ARG_{1} + \sqrt{\left|{ARG_{0}}\right|}}\right|}}}\right|} \right)} + 0


<IPython.core.display.Math object>


to_callable() at optimum (420.97, 420.97):
  Surrogate prediction : 0.0000
  True f at optimum    : 0.000025


## Run PSO on Surrogate

In [6]:
from symgene.optimization import PSOOptimizer

optimizer = PSOOptimizer(n_particles=50, n_iter=300, verbose=0)

def surrogate_fn(x):
    return float(regressor.predict(x.reshape(1, -1))[0])

pso_result = optimizer.optimize(surrogate_fn, bounds=bench.bounds, seed=0)
f_true = float(bench.fn(pso_result.x_best))

print(f'PSO found   : x={np.round(pso_result.x_best, 4)}  f_surrogate={pso_result.f_best:.4f}')
print(f'True f(x*)  : {f_true:.6f}')
print(f'True optimum: f={bench.f_opt} at x={bench.x_opt}')
print(f'Gap to optimum: {abs(f_true - bench.f_opt):.6f}')

PSO found   : x=[420.9687 420.9687]  f_surrogate=0.0000
True f(x*)  : 0.000025
True optimum: f=0.0 at x=[420.9687 420.9687]
Gap to optimum: 0.000025


## Figure 1 — Surrogate Quality: Observed × Predicted

In [ ]:
def format_r2_truncate(val, decimals=8):
    """Truncate R² to prevent 0.9999999999 from rounding to 1.00000000."""
    if val >= 1.0:
        return "1.0"
    # Format with extra digits and truncate string to avoid rounding the last digit
    s = f"{val:.14f}"
    parts = s.split(".")
    return f"{parts[0]}.{parts[1][:decimals]}"


yp_train = regressor.predict(X_train)
yp_val = regressor.predict(X_val)
yp_test = regressor.predict(X_test)

fig, ax = plt.subplots(1, 1, figsize=(7, 6))

obs_pred_ax(
    ax,
    y_train,
    yp_train,
    y_val,
    yp_val,
    y_test,
    yp_test,
    title="Surrogate Quality — Observed × Predicted",
    xlabel="Observed Schwefel f(x)",
    ylabel="Surrogate Prediction",
)

r2_tr = r2_score(y_train, yp_train)
r2_v = r2_score(y_val, yp_val)
r2_te = r2_score(y_test, yp_test)

handles, _ = ax.get_legend_handles_labels()
labels = [
    f"Train  (R²={format_r2_truncate(r2_tr, 8)})",
    f"Val    (R²={format_r2_truncate(r2_v, 8)})",
    f"Test   (R²={format_r2_truncate(r2_te, 8)})",
]

for collection in ax.collections:
    collection.set_linewidth(0)
    collection.set_edgecolor("none")

for line in ax.lines:
    if line.get_marker() != "None" and line.get_marker() != "":
        line.set_linewidth(0)
        line.set_markeredgewidth(0)

if "_snap" in globals():
    _snap(ax)

ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6, color="gray")

ax.tick_params(colors="black", which="both")
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(1.0)

leg = ax.legend(
    handles,
    labels,
    loc="upper left",
    frameon=True,
    edgecolor="black",
    fancybox=False,
    fontsize=9,
    handlelength=1.2,
    handletextpad=0.5,
)

for handle in leg.legend_handles:
    if hasattr(handle, "set_linewidth"):
        handle.set_linewidth(0)

output_path = os.path.join(FIG_DIR, "fig05_schwefel_obs_pred.png")
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {output_path}")

## Figure 2 — 3D Surface: True Schwefel vs MGGP Surrogate
Left: true Schwefel 2D analytical function (cmap `viridis`).
Right: surrogate surface learned by MGGP with custom primitives (cmap `plasma`).
The custom `xsinqrt` primitive allows the model to directly represent the Schwefel building block.

In [ ]:
def schwefel_vec(x1, x2):
    return (
        418.9829 * 2
        - x1 * np.sin(np.sqrt(np.abs(x1)))
        - x2 * np.sin(np.sqrt(np.abs(x2)))
    )


x1g = np.linspace(-500, 500, 60)
x2g = np.linspace(-500, 500, 60)
X1g, X2g = np.meshgrid(x1g, x2g)
X_grid = np.column_stack([X1g.ravel(), X2g.ravel()])

Z_true = schwefel_vec(X1g, X2g)
yp_test = regressor.predict(X_test)
Z_surr = regressor.predict(X_grid).reshape(X1g.shape)

r2_val = r2_score(y_test, yp_test)

ELEV, AZIM = 28, -55

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5.2),
    subplot_kw={"projection": "3d"},
    gridspec_kw={"wspace": -0.3},
)

plt.suptitle(
    "Schwefel 2D — True Surface vs MGGP Surrogate",
    fontsize=13,
    y=0.98,
)

panels = [
    (Z_true, "True Function", "viridis"),
    (Z_surr, f"MGGP Surrogate\nR² (test) = {r2_val:.8f}", "plasma"),
]

ticks_xy = [-500, -250, 0, 250, 500]

# --- Custom tick labels with space padding for the corner ---
# Add trailing spaces after '500' on X1 to push the label left
x_labels = ["-500", "-250", "0", "250", "500   "]

# Add leading spaces before '-500' on X2 to push the label right
y_labels = ["   -500", "-250", "0", "250", "500"]

z_min_val = 0.0
z_max_val = float(np.ceil(max(Z_true.max(), Z_surr.max()) / 100.0) * 100.0)

for ax, (Z, title, cmap) in zip(axes, panels):
    ax.computed_zorder = False

    ax.plot_surface(
        X1g,
        X2g,
        Z,
        cmap=cmap,
        alpha=0.92,
        linewidth=0,
        antialiased=True,
    )

    ax.set_title(title, fontsize=11, pad=-2)

    ax.set_xlabel("$x_1$", color="black", labelpad=4)
    ax.set_ylabel("$x_2$", color="black", labelpad=4)

    ax.set_xticks(ticks_xy)
    ax.set_xticklabels(x_labels)

    ax.set_yticks(ticks_xy)
    ax.set_yticklabels(y_labels)

    ax.zaxis.set_rotate_label(False)
    ax.set_zlabel(
        "$f(x_1,x_2)$",
        color="black",
        fontsize=9,
        labelpad=0,
        rotation=90,
        ha="right",
        va="bottom",
    )

    ax.set_zlim(z_min_val, z_max_val)
    ax.set_zticks(np.linspace(z_min_val, z_max_val, 5))

    ax.set_box_aspect(None, zoom=0.9)

    ax.view_init(elev=ELEV, azim=AZIM)
    ax.tick_params(colors="black", which="both", pad=1)

    if "_snap" in globals():
        _snap(ax)

plt.subplots_adjust(left=0.01, right=0.97, top=0.85, bottom=0.03)

output_path = os.path.join(FIG_DIR, "fig05_schwefel_surface.png")
plt.savefig(output_path, dpi=300, bbox_inches="tight", pad_inches=0.2)
plt.show()
print(f"Saved: {output_path}")

## Figure 3 — PSO Spatial Distribution: True vs Surrogate Landscape
Left: true Schwefel 2D landscape (log scale, cmap `viridis`).
Right: MGGP surrogate landscape (cmap `plasma`).
Black star = global optimum at $(420.97, 420.97)$. Red diamond = best point found by PSO on the surrogate.

In [ ]:
x1g_c = np.linspace(-500, 500, 300)
x2g_c = np.linspace(-500, 500, 300)
X1g_c, X2g_c = np.meshgrid(x1g_c, x2g_c)
X_mesh = np.column_stack([X1g_c.ravel(), X2g_c.ravel()])

Z_true_c = schwefel_vec(X1g_c, X2g_c)
Z_surr_c = regressor.predict(X_mesh).reshape(X1g_c.shape)

# Apply log1p to ensure non-negative values
Z_true_log = np.log1p(np.clip(Z_true_c, 0, None))
Z_surr_log = np.log1p(np.clip(Z_surr_c, 0, None))

# ── Unified global scale: exact top and bottom for both panels ────────────
vmin = min(Z_true_log.min(), Z_surr_log.min())
vmax = max(Z_true_log.max(), Z_surr_log.max())
levels = np.linspace(vmin, vmax, 35)  # end-to-end precision

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
plt.subplots_adjust(wspace=0.35)

panels = [
    (Z_true_log, "Schwefel 2D — True Function", "viridis"),
    (Z_surr_log, "MGGP Surrogate — Estimated Landscape", "plasma"),
]

for ax, (Z_log, title, cmap) in zip(axes, panels):
    # Contourf with levels fixed between vmin and vmax
    cp = ax.contourf(
        X1g_c,
        X2g_c,
        Z_log,
        levels=levels,
        cmap=cmap,
        alpha=0.88,
        vmin=vmin,
        vmax=vmax,
    )

    # Colorbar anchored at top and bottom
    _cb = plt.colorbar(cp, ax=ax, label="log(1 + f)", shrink=0.85)

    # Fix colorbar ticks at minimum (bottom) and maximum (top)
    cbar_ticks = np.linspace(vmin, vmax, 5)
    _cb.set_ticks(cbar_ticks)

    if "_fix_cbar" in globals():
        _fix_cbar(_cb, cp)

    # Plot optimum points
    ax.scatter(
        *bench.x_opt,
        color="black",
        marker="*",
        s=70,
        zorder=6,
        label=f"True optimum  f={bench.f_opt}",
    )
    ax.scatter(
        pso_result.x_best[0],
        pso_result.x_best[1],
        color="red",
        marker="D",
        s=15,
        zorder=7,
        edgecolors="black",
        lw=0.8,
        label=f"PSO found  f_true={f_true:.2e}",
    )

    # Standardised axis and visual formatting
    ax.set_xlabel("$x_1$", color="black", labelpad=5)
    ax.set_ylabel("$x_2$", color="black", labelpad=5)
    ax.set_title(title, fontsize=11, pad=8)

    ax.tick_params(colors="black", which="both")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1.0)

    ax.legend(
        loc="lower right",
        frameon=True,
        edgecolor="black",
        fancybox=False,
        fontsize=9,
    )

    if "_snap" in globals():
        _snap(ax)

output_path = os.path.join(FIG_DIR, "fig05_schwefel_contour.png")
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {output_path}")

## Figure 4 — PSO Convergence on the Surrogate Surface

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 6))

# Convergence curve on log scale
ax.semilogy(
    pso_result.history,
    color=TRAIN_COLOR,
    lw=2,
    label="PSO best (on surrogate)",
)

# Standardised axis labels and title
ax.set_xlabel("Iteration", color="black", labelpad=5)
ax.set_ylabel("Best fitness on surrogate (log scale)", color="black", labelpad=5)
ax.set_title("PSO Convergence on MGGP Surrogate — Schwefel 2D", fontsize=11, pad=8)

# 1. Apply _snap
if "_snap" in globals():
    _snap(ax, log_y=True)

# 2. Add grid (major and minor lines for log scale)
ax.grid(True, which="major", linestyle="--", linewidth=0.6, alpha=0.7, color="gray")
ax.grid(True, which="minor", linestyle=":", linewidth=0.4, alpha=0.4, color="gray")

# 3. Ensure solid black axes, labels and borders
ax.tick_params(colors="black", which="both")
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(1.0)

# 4. Clean legend aligned to the style
leg = ax.legend(
    loc="upper right",
    frameon=True,
    edgecolor="black",
    fancybox=False,
    fontsize=9,
    handlelength=1.2,
    handletextpad=0.5,
)

for handle in leg.legend_handles:
    if hasattr(handle, "set_linewidth"):
        handle.set_linewidth(1.5)

# 5. Save high-resolution output
output_path = os.path.join(FIG_DIR, "fig05_schwefel_pso_convergence.png")
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {output_path}")

## Numerical Results

In [11]:
print(f'Surrogate R\u00b2   (test) : {r2(y_test, yp_test):.4f}')
print(f'Surrogate MAE  (test) : {mae(y_test, yp_test):.4f}')
print(f'Surrogate MAPE (test) : {mape(y_test, yp_test):.2f}%')
print()
print(f'PSO found x       : {np.round(pso_result.x_best, 6)}')
print(f'Surrogate f(x*)   : {pso_result.f_best:.6f}')
print(f'True f(x*)        : {f_true:.6f}')
print(f'Known optimum     : f={bench.f_opt}  at x={bench.x_opt}')
print(f'Gap to optimum    : {abs(f_true - bench.f_opt):.6f}')

Surrogate R²   (test) : 1.0000
Surrogate MAE  (test) : 0.0000
Surrogate MAPE (test) : 0.00%

PSO found x       : [420.968746 420.968746]
Surrogate f(x*)   : 0.000016
True f(x*)        : 0.000025
Known optimum     : f=0.0  at x=[420.9687 420.9687]
Gap to optimum    : 0.000025
